# U.S. Electricity Generation Forecasting — SARIMA

This notebook is part of a collaborative DAEN 430 final project comparing a seasonal ARIMA model with an LSTM for monthly U.S. electricity generation forecasting.

**Authors:** Maddie Bird, Henry Supp, Jade Winebright  
**Data:** 142 monthly observations, January 1985–October 1996  
**Approach:** chronological 80/20 train/test split; seasonal ARIMA selected with `auto.arima`.



In [ ]:
# Install once if needed:
# install.packages("forecast")
library(forecast)


## Load and inspect the data

In [ ]:
df <- read.csv("data/electricity.csv")
x_vector <- as.numeric(df[[1]])
ts_data <- ts(x_vector, start = c(1985, 1), frequency = 12)

length(ts_data)
plot(ts_data,
     type = "l",
     main = "U.S. Electricity Generation (1985–1996)",
     xlab = "Time",
     ylab = "Electricity Generation")


## Chronological train/test split

In [ ]:
n <- length(ts_data)
train_size <- floor(0.8 * n)

train <- ts(ts_data[1:train_size], start = start(ts_data), frequency = 12)
test_start <- time(ts_data)[train_size + 1]
test <- ts(ts_data[(train_size + 1):n], start = test_start, frequency = 12)

c(total = n, train = length(train), test = length(test))


## Fit seasonal ARIMA and forecast

In [ ]:
arima_time <- system.time({
  fit_arima <- auto.arima(train, seasonal = TRUE)
})

fit_arima
arima_time


In [ ]:
h <- length(test)
fc_arima <- forecast(fit_arima, h = h)
accuracy(fc_arima, test)


## Forecast vs. actual

In [ ]:
options(repr.plot.width = 9, repr.plot.height = 5)

plot(fc_arima,
     fcol = "red",
     main = "ARIMA Forecast vs Actual",
     xlab = "Time",
     ylab = "Electricity Generation",
     flwd = 1)

lines(ts_data, col = "black", lwd = 1)
legend("topleft",
       legend = c("Actual", "Forecast"),
       col = c("black", "red"),
       lwd = 1,
       bty = "n")
